In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Example chemicals list
chemicals = ["Salicylic Acid", "Retinol", "Niacinamide"]

data = []

for chem in chemicals:
    # Replace spaces with '+' for search query
    search_term = chem.replace(" ", "+")
    url = f"https://www.google.com/search?q={search_term}+skincare+alternatives"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # Extract first few <p> texts as example
    paragraphs = soup.find_all("p")
    text_content = [p.get_text() for p in paragraphs[:3]]  # top 3 paragraphs

    data.append({
        "Chemical": chem,
        "Alternatives Info": " ".join(text_content)
    })

# Save dataset
df = pd.DataFrame(data)
df.to_csv("chemical_alternatives.csv", index=False)
print("Dataset saved!")


Dataset saved!


In [2]:
import re

def extract_alternatives(text):
    chemical_alts = []
    natural_alts = []
    
    # crude example: anything with 'acid', 'oxide', 'retinol' → chemical
    for word in text.split(","):
        word = word.strip()
        if re.search(r"acid|oxide|retinol|peroxide|benzoyl", word, re.I):
            chemical_alts.append(word)
        else:
            natural_alts.append(word)
    return chemical_alts, natural_alts

df["Chemical Alternatives"], df["Natural Alternatives"] = zip(*df["Alternatives Info"].apply(extract_alternatives))
df.drop(columns=["Alternatives Info"], inplace=True)
df.to_csv("structured_alternatives.csv", index=False)



In [4]:
import requests
from bs4 import BeautifulSoup

url = "https://incidecoder.com/ingredients/salicylic-acid"
res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(res.text, "html.parser")

alt_section = soup.find("div", class_="alternative-ingredients")  # depends on site
if alt_section:
    alternatives = [li.text.strip() for li in alt_section.find_all("li")]
    print(alternatives)
else:
    print("No alternatives listed")


No alternatives listed


In [5]:
import pandas as pd

# Predefined mapping of chemicals and their natural alternatives
chemicals_map = {
    "Salicylic Acid": "Willow Bark Extract, Tea Tree Oil",
    "Retinol": "Rosehip Oil, Bakuchiol",
    "Niacinamide": "Green Tea Extract, Licorice Extract",
    # Add more as needed
}

# Convert to DataFrame
df = pd.DataFrame(list(chemicals_map.items()), columns=["Chemical", "Alternatives Info"])

# Save to CSV
df.to_csv("chemicals_alternatives.csv", index=False)
print(df)


         Chemical                    Alternatives Info
0  Salicylic Acid    Willow Bark Extract, Tea Tree Oil
1         Retinol               Rosehip Oil, Bakuchiol
2     Niacinamide  Green Tea Extract, Licorice Extract
